# elutediff · 03 · Fine-tune DiffusionGemma on RT-density vectors

**Requires an A100 80GB / H100 runtime** (Runtime &rarr; Change runtime type &rarr; A100). DiffusionGemma 26B-A4B is ~52GB in bf16; the 128 MoE experts (~46GB) stay bf16, so 4-bit can't shrink them — a 40GB GPU offloads weights and is impractically slow.

This mirrors Unsloth's DiffusionGemma (26B-A4B) Sudoku notebook, swapping the Sudoku grid for the RT-density vector. The block-diffusion loop, LoRA setup, and multi-step denoising eval all come from `elutediff`.

## 1. Install Unsloth + DiffusionGemma deps

(Same incantation as the reference Unsloth notebook.)

In [ ]:
%%capture
import os, re
if 'COLAB_' not in ''.join(os.environ.keys()):
    !pip install unsloth
    !pip install --no-deps --upgrade --force-reinstall git+https://github.com/unslothai/unsloth-zoo.git git+https://github.com/unslothai/unsloth.git
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton
    !pip install --no-deps --upgrade git+https://github.com/unslothai/unsloth-zoo.git git+https://github.com/unslothai/unsloth.git
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.11.0 "tokenizers>=0.22.0,<=0.23.0"
import torch; torch._dynamo.config.recompile_limit = 64

In [ ]:
# elutediff itself (chem extra for RDKit target/feature construction).
%pip install -q "elutediff[chem] @ git+https://github.com/CodeHalwell/elutediff.git@fix/canvas-token-budget"

## 2. Load the model + attach LoRA

`FastModel` auto-detects the diffusion architecture and routes to the transformers-only slow path. LoRA targets the shared Gemma-4 backbone; the MoE experts stay frozen (~0.5% trainable params).

In [ ]:
from elutediff.config import Config
from elutediff.models.diffusion import load_model, add_lora, model_dimensions

cfg = Config()
model, processor = load_model(cfg.model)   # token=... for gated repos
print(model_dimensions(model))             # {'vocab': 262144, 'canvas_length': 256}
model = add_lora(model, cfg.model)

## 3. Build RT-density training rows

Point `METLIN_PATH` at your dataset, or use the synthetic demo so the notebook runs standalone. Re-run notebook 01 to produce a real `targets.jsonl` and upload it instead.

In [ ]:
import os, json, random
from rdkit import Chem
from rdkit.Chem import Descriptors
from elutediff.data.metlin import load_metlin
from elutediff.data.splits import make_split
from elutediff.targets.density import gaussian_density
from elutediff.targets.quantize import quantize
from elutediff.serialization.prompts import build_prompt, target_string

METLIN_PATH = '/content/demo_smrt.sdf'
if not os.path.exists(METLIN_PATH):
    rng = random.Random(0); w = Chem.SDWriter(METLIN_PATH)
    cores = ['c1ccccc1','c1ccncc1','C1CCCCC1','c1ccc2ccccc2c1','C1CCNCC1']
    for i in range(800):
        smi = 'C'*rng.randint(0,8) + rng.choice(cores) + rng.choice(['','O','N','C(=O)O','Cl','OC'])
        m = Chem.MolFromSmiles(smi)
        if m is None: continue
        rt = max(20., min(60 + 80*Descriptors.MolLogP(m) + 0.8*Descriptors.MolWt(m) + rng.uniform(-20,20), 1180.))
        m.SetProp('_Name', f'CID{i}'); m.SetProp('RETENTION_TIME', f'{rt:.1f}'); w.write(m)
    w.close()

molecules = load_metlin(METLIN_PATH)
cfg.split.strategy = 'random'   # demo only; use scaffold/cluster on real METLIN
split = make_split([m.smiles for m in molecules], cfg.split)
fold = {i:n for n,idx in (('train',split.train_idx),('val',split.val_idx),('test',split.test_idx)) for i in idx}
rows = []
for i, m in enumerate(molecules):
    lv = quantize(gaussian_density(m.rt_seconds, cfg.target), cfg.target)
    rows.append({'smiles': m.smiles, 'rt': m.rt_seconds, 'split': fold.get(i,'train'),
                 'prompt': build_prompt(smiles=m.smiles, target_cfg=cfg.target, cond_cfg=cfg.conditioning),
                 'target': target_string(lv, cfg.target)})
train_rows = [r for r in rows if r['split']=='train']
eval_rows  = [r for r in rows if r['split']=='test']
print(len(train_rows), 'train /', len(eval_rows), 'eval')
print(train_rows[0]['prompt']); print('---'); print(train_rows[0]['target'][:120], '...')

## 4. Block-diffusion fine-tuning

Pad the target to the 256-token canvas, corrupt it (replace each token with prob `t ~ U(t_lo, 1)`), and train the model to denoise back to the clean vector. Loss is cross-entropy on the target tokens + eos (padding ignored). For a real run raise `cfg.train.steps` (reference: 4000).

In [ ]:
from elutediff.training.block_diffusion import build_examples, train

cfg.train.steps = 300       # demo; raise for a real run
examples = build_examples(train_rows, processor, cfg.model)
print('usable examples:', len(examples), '/', len(train_rows))
model = train(model, examples, cfg.model, cfg.train)

## 5. Evaluate — refinement is the point

Sweep the number of denoising steps: one-shot is weak, multi-step revision is where bidirectional diffusion wins. We parse strictly, decode a point RT, and score MAE + tolerance hits + integrated window probability around the true RT.

In [ ]:
import numpy as np
from elutediff.training.sampling import generate
from elutediff.serialization.parser import parse_rt_vector, decoded_rt
from elutediff.evaluation.point_rt import point_rt_metrics, tolerance_hit_rate
from elutediff.evaluation.density import window_probability
from elutediff.targets.density import time_grid

grid = time_grid(cfg.target)
subset = eval_rows[:50]
for steps in (1, 16, 64):
    yt, yp, valid, wp = [], [], 0, []
    for r in subset:
        text = generate(model, processor, r['prompt'], steps, cfg.model.canvas_length)
        res = parse_rt_vector(text, cfg.target)
        if not res.ok: continue
        valid += 1
        yt.append(r['rt']); yp.append(decoded_rt(res.levels, cfg.target))
        wp.append(window_probability(res.levels, grid, r['rt'], cfg.target.sigma))
    if yp:
        m = point_rt_metrics(yt, yp); h = tolerance_hit_rate(yt, yp, [30,60])
        print(f"{steps:>3}-step | valid {valid}/{len(subset)} | MAE {m['mae']:.1f}s R2 {m['r2']:.3f} | "
              + ' '.join(f'{k} {v*100:.0f}%' for k,v in h.items()) + f" | window-prob {np.mean(wp):.3f}")
    else:
        print(f'{steps:>3}-step | no valid vectors parsed')

## 6. Save the LoRA adapter

In [ ]:
model.save_pretrained('diffusiongemma_rt_lora')
processor.save_pretrained('diffusiongemma_rt_lora')
# model.push_to_hub('HF_ACCOUNT/diffusiongemma_rt_lora', token='YOUR_HF_TOKEN')
# Keep the adapter unmerged for inference (merging can corrupt clipped LoRA linears).